In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.appName("MyFirstApp").master("local[*]").getOrCreate() 

In [3]:
spark

In [4]:
df = spark.read.csv("D:\\DE 202512\\CodeFiles\\Files\\sales_large.csv",header=True, inferSchema=True)

In [5]:
df.write.mode("overwrite").parquet("Files/output/")

In [6]:
df.write.mode("overwrite").parquet("Files/output/")

In [10]:
df.createOrReplaceTempView("sales")

In [11]:
query = "select sum(total_amount) as total_revenue from sales "

In [12]:
res = spark.sql(query)

In [13]:
spark.conf.set("spark.sql.shuffle.partitions",8)

In [14]:
spark.catalog.listTables()

[Table(name='sales', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

In [15]:
cust_df = spark.read.json("Files/customers_large.json")

In [16]:
cust_df = spark.read.option("multiline", "true").json("Files/customers_large.json")

In [17]:
cust_df.show()

+---------+-----------+-----------+-----------+
|     city|customer_id|       name|signup_date|
+---------+-----------+-----------+-----------+
|Hyderabad|          1| Customer_1| 2024-01-13|
|  Chennai|          2| Customer_2| 2024-05-05|
|Bangalore|          3| Customer_3| 2024-03-12|
|Hyderabad|          4| Customer_4| 2024-12-12|
|    Delhi|          5| Customer_5| 2024-02-14|
|    Delhi|          6| Customer_6| 2024-08-04|
|Hyderabad|          7| Customer_7| 2024-01-16|
|Hyderabad|          8| Customer_8| 2024-04-21|
|Bangalore|          9| Customer_9| 2024-09-15|
|    Delhi|         10|Customer_10| 2024-01-14|
|    Delhi|         11|Customer_11| 2024-04-11|
|    Delhi|         12|Customer_12| 2024-08-02|
|Bangalore|         13|Customer_13| 2024-08-17|
|    Delhi|         14|Customer_14| 2024-05-22|
|Hyderabad|         15|Customer_15| 2024-03-22|
|   Mumbai|         16|Customer_16| 2024-06-23|
|  Chennai|         17|Customer_17| 2024-03-20|
|Bangalore|         18|Customer_18| 2024

In [21]:
prod_df  = spark.read.csv("Files//products_large.csv", header=True, inferSchema=True)

In [23]:
prod_df.show()

+----------+------------+-----------+-----+
|product_id|product_name|   category|price|
+----------+------------+-----------+-----+
|       101| Product_101|Electronics|32993|
|       102| Product_102|Accessories| 8892|
|       103| Product_103|       Home| 3257|
|       104| Product_104|Accessories|77628|
|       105| Product_105|Accessories| 1442|
|       106| Product_106|Electronics| 8216|
|       107| Product_107|Accessories| 9334|
|       108| Product_108|Electronics|43809|
|       109| Product_109|Electronics|67891|
|       110| Product_110|Accessories|37000|
|       111| Product_111|    Fashion|28580|
|       112| Product_112|Accessories|75347|
|       113| Product_113|    Fashion|32350|
|       114| Product_114|    Fashion|53854|
|       115| Product_115|Accessories|12863|
|       116| Product_116|Electronics|56998|
|       117| Product_117|       Home|56019|
|       118| Product_118|    Fashion|61713|
|       119| Product_119|Electronics|13399|
|       120| Product_120|Electro

In [24]:
orders_df  = spark.read.csv("Files//orders_large.csv", header=True, inferSchema=True)

In [25]:
orders_df.show()

+--------+-----------+----------+----------+--------+
|order_id|customer_id|product_id|order_date|quantity|
+--------+-----------+----------+----------+--------+
|    1001|         77|       103|2025-02-09|       1|
|    1002|         54|       143|2025-02-07|       5|
|    1003|         67|       121|2025-03-01|       3|
|    1004|         27|       143|2025-02-15|       3|
|    1005|         31|       117|2025-01-26|       2|
|    1006|         86|       142|2025-01-20|       4|
|    1007|         41|       149|2025-03-01|       1|
|    1008|          2|       130|2025-02-09|       5|
|    1009|         13|       105|2025-02-04|       2|
|    1010|         65|       117|2025-01-09|       3|
|    1011|          9|       116|2025-01-24|       3|
|    1012|         21|       129|2025-02-23|       5|
|    1013|         91|       120|2025-02-09|       5|
|    1014|          2|       143|2025-02-22|       5|
|    1015|         39|       143|2025-01-07|       2|
|    1016|         34|      

In [27]:
orders_df.createOrReplaceTempView("orders")

In [29]:
prod_df.createOrReplaceTempView("products")

In [31]:
cust_df.createOrReplaceTempView("customers")

In [35]:
res = spark.sql("""
SELECT 
    c.name,
    p.product_name,
    SUM(o.quantity) as total_quant
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN products p ON o.product_id = p.product_id
GROUP BY c.name, p.product_name
""")

In [36]:
res

DataFrame[name: string, product_name: string, total_quant: bigint]

In [37]:
res.show()

+------------+------------+-----------+
|        name|product_name|total_quant|
+------------+------------+-----------+
| Customer_54| Product_143|          5|
| Customer_86| Product_107|          7|
| Customer_35| Product_111|          1|
| Customer_52| Product_122|          1|
| Customer_43| Product_102|          3|
| Customer_23| Product_138|          1|
| Customer_14| Product_139|          3|
| Customer_12| Product_149|          3|
| Customer_81| Product_137|          4|
|Customer_100| Product_128|          2|
| Customer_50| Product_145|          2|
| Customer_27| Product_127|          5|
| Customer_99| Product_124|          5|
| Customer_75| Product_102|          3|
| Customer_76| Product_145|          2|
| Customer_65| Product_142|          1|
| Customer_41| Product_108|          8|
| Customer_93| Product_138|          5|
| Customer_39| Product_138|          8|
| Customer_91| Product_119|          4|
+------------+------------+-----------+
only showing top 20 rows



In [38]:
spark = SparkSession.builder \
    .appName("OptimizedJob") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", 8) \
    .config("spark.executor.memory", "2g") \
    .getOrCreate()

In [39]:
df = spark.read.csv("D:\\DE 202512\\CodeFiles\\Files\\sales_large.csv",header=True, inferSchema=True)

In [45]:
spark

In [40]:
df.show()

+--------+-----------+----------+----------+--------+------------+-----------+-----+------------+
|order_id|customer_id|product_id|order_date|quantity|product_name|   category|price|total_amount|
+--------+-----------+----------+----------+--------+------------+-----------+-----+------------+
|    1001|         77|       103|2025-02-09|       1| Product_103|       Home| 3257|        3257|
|    1002|         54|       143|2025-02-07|       5| Product_143|Electronics| 7072|       35360|
|    1003|         67|       121|2025-03-01|       3| Product_121|       Home|14822|       44466|
|    1004|         27|       143|2025-02-15|       3| Product_143|Electronics| 7072|       21216|
|    1005|         31|       117|2025-01-26|       2| Product_117|       Home|56019|      112038|
|    1006|         86|       142|2025-01-20|       4| Product_142|Electronics|41604|      166416|
|    1007|         41|       149|2025-03-01|       1| Product_149|    Fashion|16213|       16213|
|    1008|          

In [43]:
df2 = df.filter(df["total_amount"] > 1000)
df3 = df2.groupBy("product_name").sum("total_amount")

In [44]:
df3.show()

+------------+-----------------+
|product_name|sum(total_amount)|
+------------+-----------------+
| Product_143|           516256|
| Product_108|          2891394|
| Product_118|          3949632|
| Product_150|          4353660|
| Product_125|          1595750|
| Product_120|          4101944|
| Product_101|          1517678|
| Product_146|          1516216|
| Product_128|          4938976|
| Product_127|           799260|
| Product_119|           763743|
| Product_121|           904142|
| Product_149|          1118697|
| Product_139|          1101817|
| Product_144|          6308855|
| Product_141|           514395|
| Product_123|          2690096|
| Product_111|          1771960|
| Product_126|          2139795|
| Product_115|           656013|
+------------+-----------------+
only showing top 20 rows



In [49]:
d = df.select("customer_id", "total_amount")

In [51]:
d.show()

+-----------+------------+
|customer_id|total_amount|
+-----------+------------+
|         77|        3257|
|         54|       35360|
|         67|       44466|
|         27|       21216|
|         31|      112038|
|         86|      166416|
|         41|       16213|
|          2|       63620|
|         13|        2884|
|         65|      168057|
|          9|      170994|
|         21|       35650|
|         91|      266360|
|          2|       35360|
|         39|       14144|
|         34|       43809|
|         96|      155019|
|         37|       62367|
|         27|      199227|
|         65|       64153|
+-----------+------------+
only showing top 20 rows



In [57]:
df_filter = df.filter(df.total_amount > 10000)

In [58]:
df_filter.show() 

+--------+-----------+----------+----------+--------+------------+-----------+-----+------------+
|order_id|customer_id|product_id|order_date|quantity|product_name|   category|price|total_amount|
+--------+-----------+----------+----------+--------+------------+-----------+-----+------------+
|    1002|         54|       143|2025-02-07|       5| Product_143|Electronics| 7072|       35360|
|    1003|         67|       121|2025-03-01|       3| Product_121|       Home|14822|       44466|
|    1004|         27|       143|2025-02-15|       3| Product_143|Electronics| 7072|       21216|
|    1005|         31|       117|2025-01-26|       2| Product_117|       Home|56019|      112038|
|    1006|         86|       142|2025-01-20|       4| Product_142|Electronics|41604|      166416|
|    1007|         41|       149|2025-03-01|       1| Product_149|    Fashion|16213|       16213|
|    1008|          2|       130|2025-02-09|       5| Product_130|Electronics|12724|       63620|
|    1010|         6

In [61]:
from pyspark.sql.functions import col

df = df.withColumn("amount_with_tax", col("total_amount") * 1.18)

In [62]:
df.show()

+--------+-----------+----------+----------+--------+------------+-----------+-----+------------+------------------+
|order_id|customer_id|product_id|order_date|quantity|product_name|   category|price|total_amount|   amount_with_tax|
+--------+-----------+----------+----------+--------+------------+-----------+-----+------------+------------------+
|    1001|         77|       103|2025-02-09|       1| Product_103|       Home| 3257|        3257|3843.2599999999998|
|    1002|         54|       143|2025-02-07|       5| Product_143|Electronics| 7072|       35360|41724.799999999996|
|    1003|         67|       121|2025-03-01|       3| Product_121|       Home|14822|       44466|          52469.88|
|    1004|         27|       143|2025-02-15|       3| Product_143|Electronics| 7072|       21216|25034.879999999997|
|    1005|         31|       117|2025-01-26|       2| Product_117|       Home|56019|      112038|         132204.84|
|    1006|         86|       142|2025-01-20|       4| Product_14

In [63]:
df.drop("price")

DataFrame[order_id: int, customer_id: int, product_id: int, order_date: date, quantity: int, product_name: string, category: string, total_amount: int, amount_with_tax: double]

In [65]:
df.withColumnRenamed("customer_id", "cust_id")

DataFrame[order_id: int, cust_id: int, product_id: int, order_date: date, quantity: int, product_name: string, category: string, price: int, total_amount: int, amount_with_tax: double]

In [67]:
df

DataFrame[order_id: int, customer_id: int, product_id: int, order_date: date, quantity: int, product_name: string, category: string, price: int, total_amount: int, amount_with_tax: double]

In [72]:
from pyspark.sql.functions import sum, avg

res = df.groupBy("product_name") \
  .agg(
      sum("total_amount").alias("total_sales"),
      avg("total_amount").alias("avg_sales")
  )

In [74]:
res.show()

+------------+-----------+------------------+
|product_name|total_sales|         avg_sales|
+------------+-----------+------------------+
| Product_143|     516256|          20650.24|
| Product_108|    2891394|         120474.75|
| Product_118|    3949632|          197481.6|
| Product_150|    4353660|256097.64705882352|
| Product_125|    1595750| 83986.84210526316|
| Product_120|    4101944|          186452.0|
| Product_101|    1517678|101178.53333333334|
| Product_146|    1516216| 79800.84210526316|
| Product_128|    4938976|235189.33333333334|
| Product_127|     799260|34750.434782608696|
| Product_119|     763743|42430.166666666664|
| Product_121|     904142| 41097.36363636364|
| Product_149|    1118697| 53271.28571428572|
| Product_139|    1101817| 64812.76470588235|
| Product_144|    6308855| 203511.4516129032|
| Product_141|     514395|           24495.0|
| Product_123|    2690096| 192149.7142857143|
| Product_111|    1771960| 98442.22222222222|
| Product_126|    2139795|164599.6

In [5]:
spark.sparkContext.setLogLevel("ERROR")

In [7]:
from pyspark.sql.types import * 

In [8]:
orders_schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("order_date", StringType(), True),
    StructField("quantity", IntegerType(), True),
])

In [14]:
orders_df  =  spark.read \
                .schema(orders_schema) \
                .option("header","true") \
                .csv("Files\orders_large.csv")

In [16]:
orders_df.show()

+--------+-----------+----------+----------+--------+
|order_id|customer_id|product_id|order_date|quantity|
+--------+-----------+----------+----------+--------+
|    1001|         77|       103|2025-02-09|       1|
|    1002|         54|       143|2025-02-07|       5|
|    1003|         67|       121|2025-03-01|       3|
|    1004|         27|       143|2025-02-15|       3|
|    1005|         31|       117|2025-01-26|       2|
|    1006|         86|       142|2025-01-20|       4|
|    1007|         41|       149|2025-03-01|       1|
|    1008|          2|       130|2025-02-09|       5|
|    1009|         13|       105|2025-02-04|       2|
|    1010|         65|       117|2025-01-09|       3|
|    1011|          9|       116|2025-01-24|       3|
|    1012|         21|       129|2025-02-23|       5|
|    1013|         91|       120|2025-02-09|       5|
|    1014|          2|       143|2025-02-22|       5|
|    1015|         39|       143|2025-01-07|       2|
|    1016|         34|      

In [17]:
orders_df = spark.read.json("Files/orders.json")

In [18]:
orders_df.show()

+------+-----------+--------+
|amount|   customer|order_id|
+------+-----------+--------+
|  2500|{101, John}|       1|
|  3000|{102, Mary}|       2|
+------+-----------+--------+



In [20]:
orders_df.select("order_id","customer.id","customer.name","amount").show()

+--------+---+----+------+
|order_id| id|name|amount|
+--------+---+----+------+
|       1|101|John|  2500|
|       2|102|Mary|  3000|
+--------+---+----+------+



In [1]:
from pyspark.sql import SparkSession
import random

spark = SparkSession.builder \
    .appName("PartitionShuffleDemo") \
    .master("local[*]") \
    .getOrCreate()

data = [
    (i, random.choice(["US","IN","UK","AU","CA"]), random.randint(100,1000))
    for i in range(1000000)
]

df = spark.createDataFrame(data, ["order_id","country","amount"])

df.show(5)

+--------+-------+------+
|order_id|country|amount|
+--------+-------+------+
|       0|     US|   233|
|       1|     AU|   714|
|       2|     US|   749|
|       3|     AU|   598|
|       4|     IN|   769|
+--------+-------+------+
only showing top 5 rows



In [2]:
df.rdd.getNumPartitions()

8

In [3]:
df2 = df.repartition(8)

In [4]:
df2.rdd.getNumPartitions()

8

In [5]:
df3 = df.coalesce(2)

In [6]:
df_hash = df.repartition(5, "country")

In [11]:
df_hash.groupBy("country").count().show()

+-------+------+
|country| count|
+-------+------+
|     CA|200126|
|     US|200017|
|     IN|200050|
|     UK|200008|
|     AU|199799|
+-------+------+



In [7]:
df_hash.groupBy("country").count().explain(True)

== Parsed Logical Plan ==
'Aggregate ['country], ['country, count(1) AS count#23L]
+- RepartitionByExpression [country#1], 5
   +- LogicalRDD [order_id#0L, country#1, amount#2L], false

== Analyzed Logical Plan ==
country: string, count: bigint
Aggregate [country#1], [country#1, count(1) AS count#23L]
+- RepartitionByExpression [country#1], 5
   +- LogicalRDD [order_id#0L, country#1, amount#2L], false

== Optimized Logical Plan ==
Aggregate [country#1], [country#1, count(1) AS count#23L]
+- RepartitionByExpression [country#1], 5
   +- Project [country#1]
      +- LogicalRDD [order_id#0L, country#1, amount#2L], false

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[country#1], functions=[count(1)], output=[country#1, count#23L])
   +- HashAggregate(keys=[country#1], functions=[partial_count(1)], output=[country#1, count#27L])
      +- Exchange hashpartitioning(country#1, 5), REPARTITION_BY_NUM, [plan_id=47]
         +- Project [country#1]
            +- Sc

In [8]:
df_range = df.repartitionByRange(5, "amount")

In [9]:
customers = [
    ("US","North America"),
    ("IN","Asia"),
    ("UK","Europe"),
    ("AU","Oceania"),
    ("CA","North America")
]

df_region = spark.createDataFrame(customers, ["country","region"])

In [10]:
joined = df.join(df_region, "country")
joined.explain(True)

== Parsed Logical Plan ==
'Join UsingJoin(Inner, [country])
:- LogicalRDD [order_id#0L, country#1, amount#2L], false
+- LogicalRDD [country#28, region#29], false

== Analyzed Logical Plan ==
country: string, order_id: bigint, amount: bigint, region: string
Project [country#1, order_id#0L, amount#2L, region#29]
+- Join Inner, (country#1 = country#28)
   :- LogicalRDD [order_id#0L, country#1, amount#2L], false
   +- LogicalRDD [country#28, region#29], false

== Optimized Logical Plan ==
Project [country#1, order_id#0L, amount#2L, region#29]
+- Join Inner, (country#1 = country#28)
   :- Filter isnotnull(country#1)
   :  +- LogicalRDD [order_id#0L, country#1, amount#2L], false
   +- Filter isnotnull(country#28)
      +- LogicalRDD [country#28, region#29], false

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [country#1, order_id#0L, amount#2L, region#29]
   +- SortMergeJoin [country#1], [country#28], Inner
      :- Sort [country#1 ASC NULLS FIRST], false, 0
      :  +- 

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import rand, floor

spark = SparkSession.builder \
    .appName("ShuffleDemo") \
    .master("local[*]") \
    .getOrCreate()

df = spark.range(0, 5_000_000)

sales_df = df.withColumn("customer_id", floor(rand()*100000)) \
    .withColumn("region_id", floor(rand()*10)) \
    .withColumn("amount", rand()*1000)

sales_df.cache()
sales_df.count()


5000000

In [5]:
sales_df.show(5)

+---+-----------+---------+-----------------+
| id|customer_id|region_id|           amount|
+---+-----------+---------+-----------------+
|  0|       1568|        3|376.0453161270533|
|  1|      76526|        3|847.7663471652437|
|  2|      63204|        6|233.8337494911411|
|  3|      36314|        7|293.1532715204564|
|  4|      34778|        7|950.1985439125577|
+---+-----------+---------+-----------------+
only showing top 5 rows



In [7]:
sales_df.groupBy("region_id").sum("amount").explain(True)

== Parsed Logical Plan ==
'Aggregate ['region_id], ['region_id, sum(amount#9) AS sum(amount)#315]
+- Project [id#0L, customer_id#2L, region_id#5L, (rand(1948154858309953357) * cast(1000 as double)) AS amount#9]
   +- Project [id#0L, customer_id#2L, FLOOR((rand(-6995707125731556720) * cast(10 as double))) AS region_id#5L]
      +- Project [id#0L, FLOOR((rand(-1059288855418199962) * cast(100000 as double))) AS customer_id#2L]
         +- Range (0, 5000000, step=1, splits=Some(8))

== Analyzed Logical Plan ==
region_id: bigint, sum(amount): double
Aggregate [region_id#5L], [region_id#5L, sum(amount#9) AS sum(amount)#315]
+- Project [id#0L, customer_id#2L, region_id#5L, (rand(1948154858309953357) * cast(1000 as double)) AS amount#9]
   +- Project [id#0L, customer_id#2L, FLOOR((rand(-6995707125731556720) * cast(10 as double))) AS region_id#5L]
      +- Project [id#0L, FLOOR((rand(-1059288855418199962) * cast(100000 as double))) AS customer_id#2L]
         +- Range (0, 5000000, step=1, split

In [11]:
from pyspark.sql.functions import *

skewed_df = sales_df.withColumn(
    "region_id",
    when(rand() < 0.8, 1).otherwise(floor(rand()*10))
)

In [12]:
skewed_df.groupBy("region_id").count().show()

+---------+-------+
|region_id|  count|
+---------+-------+
|        0|  99729|
|        7| 100038|
|        6|  99957|
|        9|  99662|
|        5|  99979|
|        1|4100003|
|        3| 100710|
|        8|  99483|
|        2| 100315|
|        4| 100124|
+---------+-------+



In [13]:
from pyspark.sql.functions import when

skewed_df = sales_df.withColumn(
    "region_id",
    when(rand() < 0.8, 1).otherwise(floor(rand()*10))
)

In [14]:
from pyspark.sql.functions import concat, lit

salted = skewed_df.withColumn(
    "salt",
    floor(rand()*5)
).withColumn(
    "region_salted",
    concat("region_id", lit("_"), "salt")
)

In [18]:
salted.groupBy("region_salted").count().orderBy("region_salted").show()

+-------------+------+
|region_salted| count|
+-------------+------+
|          0_0| 20099|
|          0_1| 20093|
|          0_2| 19831|
|          0_3| 19774|
|          0_4| 20122|
|          1_0|821124|
|          1_1|819039|
|          1_2|818252|
|          1_3|820114|
|          1_4|820232|
|          2_0| 20250|
|          2_1| 20013|
|          2_2| 20297|
|          2_3| 19994|
|          2_4| 19903|
|          3_0| 20001|
|          3_1| 20077|
|          3_2| 20021|
|          3_3| 19912|
|          3_4| 20101|
+-------------+------+
only showing top 20 rows



In [10]:
skewed_df.groupBy("region_id").count().show()

+---------+-------+
|region_id|  count|
+---------+-------+
|        0|  99796|
|        7| 100553|
|        6|  99685|
|        9|  99854|
|        5|  99877|
|        1|4101535|
|        3| 100266|
|        8|  99748|
|        2|  99286|
|        4|  99400|
+---------+-------+



In [11]:
from pyspark.sql.functions import concat, lit

salted = skewed_df.withColumn(
    "salt",
    floor(rand()*5)
).withColumn(
    "region_salted",
    concat("region_id", lit("_"), "salt")
)

In [13]:
salted.groupBy("region_salted").count().show()

+-------------+------+
|region_salted| count|
+-------------+------+
|          5_4| 19971|
|          7_2| 20236|
|          9_2| 19974|
|          5_3| 19861|
|          1_4|820474|
|          3_4| 20229|
|          1_3|820890|
|          8_1| 20109|
|          3_3| 20012|
|          2_4| 19793|
|          8_0| 20103|
|          6_4| 19480|
|          6_0| 19903|
|          0_0| 20038|
|          6_3| 20008|
|          0_1| 20132|
|          1_0|819551|
|          8_2| 19690|
|          7_1| 20056|
|          7_3| 20091|
+-------------+------+
only showing top 20 rows



In [17]:
spark.conf.set("spark.sql.shuffle.partitions", 200)
sales_df.groupBy("region_id").sum("amount").count()

10

In [16]:
spark.conf.set("spark.sql.shuffle.partitions", 8)
sales_df.groupBy("region_id").sum("amount").count()

10

In [18]:
sales_df.is_cached

True

In [19]:
sales_df.groupBy("region_id").sum("amount").show()
sales_df.groupBy("region_id").avg("amount").show()

+---------+--------------------+
|region_id|         sum(amount)|
+---------+--------------------+
|        0|2.4940503262854418E8|
|        7| 2.502495007891633E8|
|        6|2.5040397522270212E8|
|        9|2.4952239277047858E8|
|        5|2.4984772305462578E8|
|        1| 2.503174052778917E8|
|        3| 2.504612049043876E8|
|        8|2.4973903012655237E8|
|        2|2.5014759080562946E8|
|        4| 2.496774124407687E8|
+---------+--------------------+

+---------+------------------+
|region_id|       avg(amount)|
+---------+------------------+
|        0| 499.4973736432214|
|        7|500.81251020971786|
|        6|499.75446802680375|
|        9|499.71940674005623|
|        5|499.27804965973604|
|        1| 500.3826177164543|
|        3| 500.3510089565468|
|        8| 500.1733011416917|
|        2| 499.8493160213679|
|        4|499.72462110115225|
+---------+------------------+



In [21]:
sales_df.cache()
sales_df.count()

5000000

In [22]:
sales_df.groupBy("region_id").sum("amount").show()
sales_df.groupBy("region_id").avg("amount").show()

+---------+--------------------+
|region_id|         sum(amount)|
+---------+--------------------+
|        0|2.4940503262854418E8|
|        7| 2.502495007891633E8|
|        6|2.5040397522270212E8|
|        9|2.4952239277047858E8|
|        5|2.4984772305462578E8|
|        1| 2.503174052778917E8|
|        3| 2.504612049043876E8|
|        8|2.4973903012655237E8|
|        2|2.5014759080562946E8|
|        4| 2.496774124407687E8|
+---------+--------------------+

+---------+------------------+
|region_id|       avg(amount)|
+---------+------------------+
|        0| 499.4973736432214|
|        7|500.81251020971786|
|        6|499.75446802680375|
|        9|499.71940674005623|
|        5|499.27804965973604|
|        1| 500.3826177164543|
|        3| 500.3510089565468|
|        8| 500.1733011416917|
|        2| 499.8493160213679|
|        4|499.72462110115225|
+---------+------------------+



In [23]:
sales_df.cache()

DataFrame[id: bigint, customer_id: bigint, region_id: bigint, amount: double]

In [24]:
sales_df.unpersist()

DataFrame[id: bigint, customer_id: bigint, region_id: bigint, amount: double]

In [25]:
sales_df.groupBy("region_id").sum("amount").show()
sales_df.groupBy("region_id").avg("amount").show()

+---------+--------------------+
|region_id|         sum(amount)|
+---------+--------------------+
|        0|2.4940503262854418E8|
|        7| 2.502495007891633E8|
|        6|2.5040397522270212E8|
|        9|2.4952239277047858E8|
|        5|2.4984772305462578E8|
|        1| 2.503174052778917E8|
|        3| 2.504612049043876E8|
|        8|2.4973903012655237E8|
|        2|2.5014759080562946E8|
|        4| 2.496774124407687E8|
+---------+--------------------+

+---------+------------------+
|region_id|       avg(amount)|
+---------+------------------+
|        0| 499.4973736432214|
|        7|500.81251020971786|
|        6|499.75446802680375|
|        9|499.71940674005623|
|        5|499.27804965973604|
|        1| 500.3826177164543|
|        3| 500.3510089565468|
|        8| 500.1733011416917|
|        2| 499.8493160213679|
|        4|499.72462110115225|
+---------+------------------+



In [26]:
sales_df.cache()

DataFrame[id: bigint, customer_id: bigint, region_id: bigint, amount: double]

In [27]:
sales_df.groupBy("region_id").sum("amount").show()
sales_df.groupBy("region_id").avg("amount").show()

+---------+--------------------+
|region_id|         sum(amount)|
+---------+--------------------+
|        0|2.4940503262854418E8|
|        7| 2.502495007891633E8|
|        6|2.5040397522270212E8|
|        9|2.4952239277047858E8|
|        5|2.4984772305462578E8|
|        1| 2.503174052778917E8|
|        3| 2.504612049043876E8|
|        8|2.4973903012655237E8|
|        2|2.5014759080562946E8|
|        4| 2.496774124407687E8|
+---------+--------------------+

+---------+------------------+
|region_id|       avg(amount)|
+---------+------------------+
|        0| 499.4973736432214|
|        7|500.81251020971786|
|        6|499.75446802680375|
|        9|499.71940674005623|
|        5|499.27804965973604|
|        1| 500.3826177164543|
|        3| 500.3510089565468|
|        8| 500.1733011416917|
|        2| 499.8493160213679|
|        4|499.72462110115225|
+---------+------------------+



In [28]:
import requests
import zipfile
import io

KAGGLE_USERNAME = "andepai"
KAGGLE_KEY = "KGAT_cbdfaeca4f3539f446d69279e3fd9412"

owner = "andepai"
dataset = "orders"

url = f"https://www.kaggle.com/api/v1/datasets/download/{owner}/{dataset}"

response = requests.get(url, auth=(KAGGLE_USERNAME, KAGGLE_KEY), stream=True)

if response.status_code == 200:
    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        z.extractall("./downloaded_data")
    print("Download and extraction complete.")
else:
    print(f"Failed to download: {response.status_code} - {response.text}")


Download and extraction complete.


In [1]:
import pandas as pd

In [3]:
data = [(1,"Alice","North",1200),
        (2,"Bob","South",None),
        (3,"Charlie",None,1800),
        (4,"David","West",2000)
        ]
columns = ["CustomerID", "CustomerName", "Region", "SalesAmount"]

In [8]:
df = pd.DataFrame(data,columns=columns)

In [9]:
df

,CustomerID,CustomerName,Region,SalesAmount
0,1,Alice,North,1200.0
1,2,Bob,South,NaN
2,3,Charlie,None,1800.0
3,4,David,West,2000.0


In [1]:
import snowflake.connector

In [2]:
import pandas  as pd

In [3]:
conn = snowflake.connector.connect(
    user='detrainings01',
    password='DataEngineering@2026',
    account='sqqpjbk-pi02317',  # like abc-xy12345
    warehouse='COMPUTE_WH',
    database='MY_DB1',
    schema='PUBLIC'
)

In [5]:
cur = conn.cursor()

In [6]:
cur.execute("SELECT CURRENT_USER(), CURRENT_VERSION()")

In [7]:
print(cur.fetchall())

[('DETRAININGS01', '10.6.2')]


In [8]:
cur.execute("""
CREATE OR REPLACE TABLE EMPLOYEES (
    ID INT,
    NAME STRING,
    SALARY NUMBER
)
""")

In [9]:
cur.execute("""
INSERT INTO EMPLOYEES VALUES
(1, 'John', 50000),
(2, 'Mary', 60000)
""")

In [10]:
cur.execute("SELECT * FROM EMPLOYEES")
rows = cur.fetchall()

for row in rows:
    print(row)

(1, 'John', 50000)
(2, 'Mary', 60000)


In [11]:
cur.execute("SELECT * FROM EMPLOYEES")
print(cur.fetchone())

(1, 'John', 50000)


In [12]:
cur.execute("SELECT * FROM EMPLOYEES")
print(cur.fetchmany(1))

[(1, 'John', 50000)]


In [13]:
id =1 
cur.execute(f"SELECT * FROM EMPLOYEES WHERE ID = {id}")

In [14]:
cur.fetchone()

(1, 'John', 50000)

In [17]:
query = "SELECT * FROM EMPLOYEES"
df = pd.read_sql(query, conn)

C:\Users\Chinna\AppData\Local\Temp\ipykernel_11484\3481527895.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [18]:
df

,ID,NAME,SALARY
0,1,John,50000
1,2,Mary,60000


In [4]:
new_emp = pd.DataFrame({"ID":[3], "NAME":["Alice"], "SALARY":[55000]})

In [5]:
new_emp

,ID,NAME,SALARY
0,3,Alice,55000


In [6]:
from snowflake.connector.pandas_tools import write_pandas

In [33]:
import sys
print(sys.version)

3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]


In [7]:
success, nchunks, nrows, _ = write_pandas(
    conn,
    new_emp,
    "EMPLOYEES"
)